# Cell-of-origin correlation analysis: healthy vs cancer

For the `healthy` and `cancer` samples in the pipeline's `results/` folder:

1. loads the per-gene FFT WPS matrix (`results/<sample>/proj/fft_summaries/fft_<sample>_WPS.tsv.gz`, genes x periods 120-280 bp)
2. scores each gene as the mean FFT intensity over 193-199 bp (same as `collapse_fft_wps.py`)
3. correlates the scores with the Tabula Sapiens reference using several methods and compares them visually
4. plots correlation-vs-period curves ("S curves") per sample and healthy vs cancer

**Colour code used in every plot:** red = blood/immune groups, grey = all other groups,
black = erythroid groups (erythrocytes and erythroid progenitors). Erythroid groups are kept out of the blood/immune
set: mature erythrocytes have no nucleus and cannot release cfDNA, so they act as a negative control and should
rank near the bottom / stay flat.

## Settings

In [ ]:
from pathlib import Path

# the pipeline's workflow/ directory; ".." works when this notebook sits in workflow/result_analysis/
WORKFLOW_DIR = Path("..")

REF_CSV = WORKFLOW_DIR / "resources/avg_expr_by_celltype_tissue.csv"
RESULTS_DIR = WORKFLOW_DIR / "results"
SAMPLES = ["healthy", "cancer"]

# where plots and the two summary tables are written
OUT_DIR = Path("output")

BAND = (193, 199)                               # period band (bp) used for the gene score
VAR_GENE_COUNTS = [5000, 3000, 2000, 1000, 500] # Pearson on top-N variable genes
CURVE_METHODS = ["baseline", "pearson_var5k", "pearson_var3k", "pearson_var2k"]  # methods drawn as S curves
TOP_N = 20

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display

OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 45)
pd.set_option("display.width", 250)

def fft_path(sample):
    d = RESULTS_DIR / sample / "proj" / "fft_summaries"
    for name in (f"fft_{sample}_WPS.tsv.gz", f"fft_{sample}_WPS.tsv"):
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"no FFT matrix for '{sample}' in {d}")

assert REF_CSV.exists(), f"reference not found: {REF_CSV}"
FFT_FILES = {s: fft_path(s) for s in SAMPLES}
for s, p in FFT_FILES.items():
    print(f"{s}: {p}")
print("output:", OUT_DIR.resolve())

# one colour per category and per sample, used by every plot below
CAT_COLORS = {"blood/immune": "tab:red", "other": "tab:gray", "erythroid": "black"}
SAMPLE_COLORS = {"healthy": "tab:blue", "cancer": "tab:orange"}

## Load data

In [ ]:
ref = pd.read_csv(REF_CSV, index_col=0)
gene_names = ref.pop("gene_name")
print(f"reference: {ref.shape[0]} genes x {ref.shape[1]} groups")

IMMUNE = ["monocyte", "macrophage", "T cell", "B cell", "neutrophil", "dendritic", "NK", "natural killer",
          "lymphocyte", "leukocyte", "myeloid", "plasma cell", "mast cell", "thymocyte", "hematopoietic",
          "platelet", "megakaryocyte", "basophil", "eosinophil", "granulocyte", "microglial"]

def category(group):
    if group.startswith(("erythrocyte", "erythroid")):
        return "erythroid"
    if any(k.lower() in group.lower() for k in IMMUNE):
        return "blood/immune"
    return "other"

cat = pd.Series({g: category(g) for g in ref.columns})
cat_color = cat.map(CAT_COLORS)
print(cat.value_counts().to_string())

def load_fft(path):
    fft = pd.read_csv(path, sep="\t", index_col=0)
    fft.columns = fft.columns.astype(float)
    return fft[sorted(fft.columns)]

fft = {s: load_fft(p) for s, p in FFT_FILES.items()}
scores = {}
for s, m in fft.items():
    band_cols = [c for c in m.columns if BAND[0] <= c <= BAND[1]]
    scores[s] = m[band_cols].mean(axis=1)
    print(f"{s}: {m.shape[0]} genes x {m.shape[1]} periods, band {band_cols}, "
          f"{len(ref.index.intersection(m.index))} genes shared with reference, median score {scores[s].median():.1f}")

## Methods

- **baseline**: Pearson, all shared genes (what the pipeline's `correlate.py` does)
- **pearson_varN**: Pearson on the N genes whose expression varies most across groups (drops genes that are flat
  across cell types and can't distinguish them)
- **spearman_var5k**: Spearman on the top 5000 variable genes, for comparison only: rank ties on very sparse groups
  such as `erythrocyte_Blood` push them to the top

In [ ]:
def variable_genes(expr, n):
    return expr.var(axis=1).nlargest(n).index

def label(n):
    return f"pearson_var{n // 1000}k" if n >= 1000 and n % 1000 == 0 else f"pearson_var{n}"

def gene_set(method, expr):  # genes a method correlates over
    if method == "baseline":
        return expr.index
    n = 5000 if method.startswith("spearman") else int(method.split("var")[1].replace("k", "000"))
    return variable_genes(expr, n)

METHODS = ["baseline"] + [label(n) for n in VAR_GENE_COUNTS] + ["spearman_var5k"]

def correlate(method, expr, score):
    g = gene_set(method, expr)
    e, s = expr.loc[g], score.loc[g]
    if method.startswith("spearman"):
        return e.rank().corrwith(s.rank())
    return e.corrwith(s)

results = {}  # (method, sample) -> correlation per group
for s in SAMPLES:
    shared = ref.index.intersection(scores[s].index)
    for m in METHODS:
        results[(m, s)] = correlate(m, ref.loc[shared], scores[s].loc[shared]).sort_values()
print("methods:", METHODS)

## Method summary

In [ ]:
ERY = cat.index[cat == "erythroid"]
rows = []
for m in METHODS:
    for s in SAMPLES:
        r = results[(m, s)]
        rk = r.rank()
        rows.append({"method": m, "sample": s, "min_r": r.min(), "spread": r.max() - r.min(),
                     f"immune_top{TOP_N}": (cat[r.index[:TOP_N]] == "blood/immune").mean(),
                     "erythroid_median_rank": rk[ERY].median(),
                     "erythrocyte_Blood_rank": int(rk["erythrocyte_Blood"]), "top1": r.index[0]})
    rows[-1]["healthy_vs_cancer_rho"] = rows[-2]["healthy_vs_cancer_rho"] = \
        results[(m, "healthy")].corr(results[(m, "cancer")], method="spearman")
summary = pd.DataFrame(rows).set_index(["method", "sample"])
summary.to_csv(OUT_DIR / "method_summary.csv")
summary.round(3)

In [ ]:
# method comparison at a glance
n_groups = ref.shape[1]
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
x = np.arange(len(METHODS)); w = 0.38
for i, s in enumerate(SAMPLES):
    sm = summary.xs(s, level="sample").loc[METHODS]
    off = (i - 0.5) * w
    axes[0].bar(x + off, sm["min_r"], w, color=SAMPLE_COLORS[s], label=s)
    axes[1].bar(x + off, sm["spread"], w, color=SAMPLE_COLORS[s])
    axes[2].bar(x + off, sm[f"immune_top{TOP_N}"], w, color=SAMPLE_COLORS[s])
    axes[3].bar(x + off, sm["erythroid_median_rank"], w, color=SAMPLE_COLORS[s])
rho = summary.xs("healthy", level="sample").loc[METHODS, "healthy_vs_cancer_rho"]
axes[1].plot(x, rho * summary["spread"].max(), "k.--", label="healthy vs cancer ρ (scaled)")
axes[3].axhline(n_groups, color="k", ls=":", lw=1); axes[3].axhline(n_groups / 2, color="k", ls=":", lw=0.6)
titles = ["strongest correlation (min r) - lower is stronger", "spread (max − min r) - higher separates more",
          f"fraction of top {TOP_N} that is blood/immune", f"erythroid median rank - higher is better (max {n_groups})"]
for ax, t in zip(axes, titles):
    ax.set_xticks(x); ax.set_xticklabels(METHODS, rotation=40, ha="right"); ax.set_title(t, fontsize=10)
axes[0].legend(); axes[1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "method_comparison.png", dpi=150)
plt.show()

## Healthy vs cancer, per method

Each dot is a cell-type group: x = correlation in healthy, y = correlation in cancer; dashed line = equal in both.
A tight diagonal cloud = consistent method; the whole cloud off the diagonal = global (likely technical, e.g. depth)
shift; single groups far off the line = candidate sample-specific signal (needs replicates to be meaningful).

In [ ]:
cat_legend = [Line2D([], [], marker="o", ls="", color=c, label=k) for k, c in CAT_COLORS.items()]
order = cat.sort_values(key=lambda c: c.map({"other": 0, "blood/immune": 1, "erythroid": 2})).index  # draw erythroid on top

ncol = 4
nrow = int(np.ceil(len(METHODS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.8 * nrow))
for ax, m in zip(axes.flat, METHODS):
    h, c = results[(m, "healthy")][order], results[(m, "cancer")][order]
    ax.scatter(h, c, s=np.where(cat[order] == "erythroid", 40, 10), c=cat_color[order], alpha=0.7)
    lo, hi = min(h.min(), c.min()), max(h.max(), c.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    ax.set_title(f"{m}  (ρ = {h.corr(c, method='spearman'):.2f})")
    ax.set_xlabel("healthy r"); ax.set_ylabel("cancer r")
for ax in list(axes.flat)[len(METHODS):]:
    ax.axis("off")
axes.flat[0].legend(handles=cat_legend, fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "scatter_healthy_vs_cancer.png", dpi=150)
plt.show()

## Correlations by cell-type category

Distribution of every group's correlation, split by category, for each method and sample. A useful method separates
blood/immune (more negative) from other groups, and keeps erythroid groups at or above the "other" level.

In [ ]:
cats = list(CAT_COLORS)
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.2 * nrow), sharey=False)
for ax, m in zip(axes.flat, METHODS):
    for i, k in enumerate(cats):
        for j, s in enumerate(SAMPLES):
            vals = results[(m, s)][cat[results[(m, s)].index] == k]
            pos = i * 3 + j
            bp = ax.boxplot(vals, positions=[pos], widths=0.8, patch_artist=True, showfliers=False)
            bp["boxes"][0].set(facecolor=SAMPLE_COLORS[s], alpha=0.5, edgecolor=CAT_COLORS[k])
            ax.scatter(np.random.default_rng(0).normal(pos, 0.08, len(vals)), vals, s=5, color=CAT_COLORS[k], alpha=0.5)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xticks([i * 3 + 0.5 for i in range(len(cats))]); ax.set_xticklabels(cats)
    ax.set_title(m); ax.set_ylabel("r")
for ax in list(axes.flat)[len(METHODS):]:
    ax.axis("off")
axes.flat[0].legend(handles=[Patch(color=c, alpha=0.5, label=s) for s, c in SAMPLE_COLORS.items()], fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "category_distributions.png", dpi=150)
plt.show()

## Top groups (bars coloured by category)

In [ ]:
SHOW = [m for m in CURVE_METHODS if m in METHODS]
fig, axes = plt.subplots(len(SAMPLES), len(SHOW), figsize=(5.5 * len(SHOW), 0.33 * TOP_N * len(SAMPLES) + 1.5))
for i, s in enumerate(SAMPLES):
    for j, m in enumerate(SHOW):
        ax = axes[i, j]
        top = results[(m, s)].iloc[:TOP_N][::-1]
        ax.barh(range(len(top)), top.values, color=cat_color[top.index])
        ax.set_yticks(range(len(top))); ax.set_yticklabels([g[:40] for g in top.index], fontsize=7)
        ax.set_title(f"{s}: {m}", fontsize=10); ax.set_xlabel("r")
axes[0, 0].legend(handles=[Patch(color=c, label=k) for k, c in CAT_COLORS.items()], fontsize=8, loc="lower left")
plt.tight_layout()
fig.savefig(OUT_DIR / "top_groups.png", dpi=150)
plt.show()

In [ ]:
# the same top lists as a table, erythroid groups marked with *
for s in SAMPLES:
    top = pd.DataFrame({m: [g + (" *" if cat[g] == "erythroid" else "") for g in results[(m, s)].index[:TOP_N]]
                        for m in METHODS})
    top.index = range(1, TOP_N + 1)
    print(f"=== {s}")
    display(top)

In [ ]:
allcorr = pd.DataFrame({f"{m}__{s}": r for (m, s), r in results.items()})
allcorr.insert(0, "category", cat)
allcorr.index.name = "cell_type"
allcorr.to_csv(OUT_DIR / "correlations_all_methods.csv")
print(f"wrote {OUT_DIR / 'correlations_all_methods.csv'}")

## S curves

For every period column (120-280 bp), correlate each gene's FFT intensity at that period with each group's
expression, using the same gene set as the method. Lines are the mean over each category's groups (shaded = ±1 SD
across groups). Contributing cell types should dip around 193-199 bp (gold band); erythroid should stay flat.

In [ ]:
def period_curves(method, m):
    shared = ref.index.intersection(m.index)
    g = gene_set(method, ref.loc[shared])
    expr = ref.loc[g]
    return pd.DataFrame({p: expr.corrwith(m.loc[g, p]) for p in m.columns}).T  # periods x groups

curves = {(meth, s): period_curves(meth, fft[s]) for meth in CURVE_METHODS for s in SAMPLES}

def cat_mean(curve, k):
    sub = curve.loc[:, cat[curve.columns] == k]
    return sub.mean(axis=1), sub.std(axis=1)

def band(ax):
    ax.axvspan(*BAND, color="gold", alpha=0.25)
    ax.axhline(0, color="k", lw=0.5)

In [ ]:
# per sample: the three category means
fig, axes = plt.subplots(len(CURVE_METHODS), len(SAMPLES), figsize=(7 * len(SAMPLES), 3.6 * len(CURVE_METHODS)),
                         sharex=True, squeeze=False)
for i, meth in enumerate(CURVE_METHODS):
    for j, s in enumerate(SAMPLES):
        ax = axes[i, j]; band(ax)
        c = curves[(meth, s)]
        for k, col in CAT_COLORS.items():
            mu, sd = cat_mean(c, k)
            ax.plot(mu.index, mu, color=col, lw=2, ls="--" if k == "erythroid" else "-", label=k)
            ax.fill_between(mu.index, mu - sd, mu + sd, color=col, alpha=0.08)
        ax.set_title(f"{s}: {meth}", fontsize=10); ax.set_ylabel("r")
for ax in axes[-1]:
    ax.set_xlabel("period (bp)")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "scurves_per_sample.png", dpi=150)
plt.show()

In [ ]:
# healthy vs cancer: one panel per category, both samples overlaid
cats = list(CAT_COLORS)
fig, axes = plt.subplots(len(CURVE_METHODS), len(cats), figsize=(6 * len(cats), 3.6 * len(CURVE_METHODS)),
                         sharex=True, squeeze=False)
for i, meth in enumerate(CURVE_METHODS):
    for j, k in enumerate(cats):
        ax = axes[i, j]; band(ax)
        for s in SAMPLES:
            mu, sd = cat_mean(curves[(meth, s)], k)
            ax.plot(mu.index, mu, color=SAMPLE_COLORS[s], lw=2, label=s)
            ax.fill_between(mu.index, mu - sd, mu + sd, color=SAMPLE_COLORS[s], alpha=0.12)
        ax.set_title(f"{meth}: {k} mean", fontsize=10, color=CAT_COLORS[k] if k != "other" else "dimgray")
        ax.set_ylabel("r")
for ax in axes[-1]:
    ax.set_xlabel("period (bp)")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "scurves_healthy_vs_cancer.png", dpi=150)
plt.show()

In [ ]:
# blood/immune minus other: the cell-type-specific part of the dip, healthy vs cancer, all curve methods
fig, axes = plt.subplots(1, len(CURVE_METHODS), figsize=(5 * len(CURVE_METHODS), 3.8), sharey=True)
for ax, meth in zip(np.atleast_1d(axes), CURVE_METHODS):
    band(ax)
    for s in SAMPLES:
        c = curves[(meth, s)]
        ax.plot(c.index, cat_mean(c, "blood/immune")[0] - cat_mean(c, "other")[0], color=SAMPLE_COLORS[s], lw=2, label=s)
    ax.set_title(meth, fontsize=10); ax.set_xlabel("period (bp)")
np.atleast_1d(axes)[0].set_ylabel("r(blood/immune) − r(other)")
np.atleast_1d(axes)[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "scurves_immune_minus_other.png", dpi=150)
plt.show()